# Which frozen DNA language model is best for splice sites?

Four pre-trained DNA language models, the same data, the same small classifier on top. The models are **frozen**: nothing inside them is trained here. Each model only turns every 600-letter DNA window into a vector of numbers (an *embedding*), and a small network (the *head*) learns to read that vector and say **donor**, **acceptor** or **none**.

What happens, in order:

1. Load the splice-site windows: all ~30,000 training windows and all ~3,000 test windows (chr20/21 are test only).
2. For each model: find the token that covers the centre of the window, run the model once over every window, and keep two vectors per window (the centre token and the token before it, glued together). Save them to disk straight away.
3. Train the same head on those vectors, for each model, and pick the best epoch on a 10% validation part of the training data.
4. Score each model on the test windows and write `results.json`, one comparison chart, one confusion-matrix figure per model, and zip files to download.

Self-contained: everything it needs from the local repo (`02_dna-splicing-pipeline/scripts/`) is copied in below, and each piece says which file it came from.

**Kaggle settings:** Accelerator = **GPU T4 x2** (only the first GPU is used), Internet = **On**.
First run it with `SMOKE = True` (a few minutes, one small model) to see that it works. Then set `SMOKE = False` and use **Save Version → Save & Run All (Commit)** for the full run, so it keeps running when the browser tab is closed.

**Expected time for the full run on a T4:** about 1 hour, somewhere between 45 and 100 minutes. This is an estimate, not a measurement: our earlier Kaggle run embedded about 390 windows per second with NT-v2 50M, and the bigger models are scaled from that by their size. Most of the time is the 2.5B model (roughly 25 to 70 minutes with its 10 GB download). The notebook prints how long each model really takes.

**Checked on the laptop (CPU) before upload:** the SMOKE run from top to bottom; the centre-token lookup on all 33,000 windows with all four tokenizers; NT-v2 50M and DNABERT-2 vectors against the local pipeline's saved ones (same to float16 rounding); half precision with NT-v2 50M (cosine with full precision 0.99999 on 16 windows); the DNABERT-2 `triton` fix with a stand-in `triton` installed; the out-of-memory back-off with a simulated error; skipping a model that cannot load; reusing saved embeddings on a second run.

**Not checked, because the laptop has no GPU:** anything on a real GPU (speed, memory, half precision there), the NT-v2 500M and NT 2.5B weights (only their tokenizers and settings were loaded), and Kaggle's real `triton`. The first full run is the test of those; if a model fails, it is skipped with the reason in `results.json`.

In [ ]:
# Same pin as dna-venv (scripts/setup_local_env.sh). Must run before anything imports transformers.
%pip install -q transformers==4.49.0 einops

## Settings

Everything you might want to change is in the next cell. `SMOKE = True` is a quick test: 200 training windows, 100 test windows, and only the smallest model. Its numbers mean nothing; it only shows that the notebook runs from top to bottom.

The models run from smallest to largest, so the cheap ones are finished and saved before the big one starts.

In [ ]:
SMOKE = True   # True = 200 train / 100 test windows, NT-v2 50M only (does it run?). False = the full comparison.

# The models, in the order they run (smallest first). All frozen, inference only.
#   revision    = the exact commit of the model repo, so a later upload cannot change the result
#   weights     = "fp32": weights stay full precision on the GPU and the arithmetic runs in half precision
#                 (fp16 autocast, as in our earlier Kaggle notebooks).
#                 "fp16": the weights themselves are loaded in half precision. Only for the 2.5B model:
#                 its fp32 weights are 10 GB and would nearly fill the 15 GB T4.
#   download_gb = size of the weight files, used to check there is disk space before downloading
MODELS = [
    {"key": "nt_v2_50m", "name": "NT-v2 50M",
     "repo": "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species",
     "revision": "81b29e5786726d891dbf929404ef20adca5b36f1",       # the snapshot dna-venv has cached
     "weights": "fp32", "download_gb": 0.3},
    {"key": "dnabert2_117m", "name": "DNABERT-2 117M",
     "repo": "zhihan1996/DNABERT-2-117M",
     "revision": "7bce263b15377fc15361f52cfab88f8b586abda0",       # the snapshot dna-venv has cached
     "weights": "fp32", "download_gb": 0.5,
     "block_triton": True},                                        # see "DNABERT-2 and triton" below
    {"key": "nt_v2_500m", "name": "NT-v2 500M",
     "repo": "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species",
     "revision": "06615c1660c892fc199840c18123f8385b3542a8",
     "weights": "fp32", "download_gb": 2.0},
    {"key": "nt_2500m", "name": "NT 2.5B",
     "repo": "InstaDeepAI/nucleotide-transformer-2.5b-multi-species",
     "revision": "b746b125aacd1b0970c05a32fd71ba726754542e",
     "weights": "fp16", "download_gb": 10.2},
]
REFERENCE = "nt_v2_50m"           # the model from our earlier runs; the others are compared with it
SMOKE_MODELS = ["nt_v2_50m"]

DATASET_REPO = "InstaDeepAI/nucleotide_transformer_downstream_tasks_revised"   # phase1_common.py
DATASET_CONFIG = "splice_sites_all"                                            # phase1_common.py
DATASET_REVISION = "851f9946252e90c665cdb3cc3eedb78f1f26197c"   # data/label_info.json: dataset_commit

SEED = 42                # sampling seed (phase1_common.SEED); only used by the SMOKE sample
HEAD_SEED = 0            # 03_train_head.py --seed: validation split and head initialisation
N_TRAIN, N_TEST = (200, 100) if SMOKE else (-1, -1)     # -1 = every window
VAL_FRAC = 0.10          # 03_train_head.py --val-frac
CENTER_BASE = 300        # 02b_embed_center.py / 12_repool_center.py

EMBED_BATCH = 64         # windows per forward pass; halved automatically if the GPU runs out of memory
MIN_BATCH = 4            # give up on a model if even this does not fit
FP16_MIN_COSINE = 0.99   # first batch: fp16 vectors must point the same way as fp32 vectors
FREE_MODEL_CACHE = True  # on Kaggle only: delete each model's download once its embeddings are saved

# Head recipe = 03_train_head.py defaults (the recipe behind the local 0.9405).
HEAD_LR, HEAD_WEIGHT_DECAY, HEAD_BATCH, HEAD_MAX_EPOCHS, HEAD_PATIENCE = 1e-3, 0.0, 128, 300, 20

N_BOOT = 2000            # bootstrap resamples of the test windows, for the 95% intervals

# data/label_info.json (worked out from the motifs by 01_get_data.py; re-checked below)
LABEL_NAMES = {0: "none", 1: "acceptor", 2: "donor"}
MOTIF_OFFSETS = {"donor_GT_plus": 301, "donor_AC_minus": 298,
                 "acceptor_AG_plus": 298, "acceptor_CT_minus": 301}
TRAIN_CHROMS = {str(i) for i in range(1, 20)} | {"X", "Y"}
TEST_CHROMS = {"20", "21"}

In [ ]:
import collections, contextlib, copy, gc, hashlib, json, os, platform, shutil, sys, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from torch import nn

T_START = time.time()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")   # one GPU, no DataParallel
ON_GPU = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if ON_GPU else None
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if ON_GPU else None

ON_KAGGLE = os.path.isdir("/kaggle/working")
BASE_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
OUT_DIR = BASE_DIR / ("compare_models_smoke" if SMOKE else "compare_models")   # SMOKE never mixes with the full run
EMB_DIR, FIG_DIR, PRED_DIR = OUT_DIR / "embeddings", OUT_DIR / "figures", OUT_DIR / "predictions"
for d in (EMB_DIR, FIG_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)
RESULTS_JSON = OUT_DIR / "results.json"

RUN_MODELS = [m for m in MODELS if not SMOKE or m["key"] in SMOKE_MODELS]
assert len({m["key"] for m in MODELS}) == len(MODELS) and any(m["key"] == REFERENCE for m in MODELS)

print(f"python {platform.python_version()}  torch {torch.__version__}  "
      f"transformers {transformers.__version__}  sklearn {sklearn.__version__}")
print(f"device: {DEVICE}" + (f" ({GPU_NAME}, {GPU_TOTAL_GB:.1f} GB)" if ON_GPU else "")
      + f"   half precision (fp16): {ON_GPU}")
print(f"SMOKE = {SMOKE}: models {[m['name'] for m in RUN_MODELS]}")
print(f"output folder: {OUT_DIR}")
assert transformers.__version__ == "4.49.0", "restart the kernel so the pinned transformers is the one imported"
if not SMOKE and not ON_GPU:
    print("WARNING: the full run on a CPU would take days. On Kaggle: Settings -> Accelerator -> GPU T4 x2.")

## Data: same files, same chromosome split, all windows

The dataset comes as two files. The training file holds windows from chromosomes 1–19, X and Y; the test file holds windows from chromosomes 20 and 21 only. That way the test never contains a piece of DNA the head has seen.

The earlier runs used a sample of 8,000 + 2,000. This one uses every window. One tenth of the training windows is set aside as **validation**: the head never trains on those, and they decide when to stop training. The test windows are only used once, at the end.

The cell also re-checks what the label numbers mean, the way `01_get_data.py` found it: donors carry `GT` just after the centre, acceptors carry `AG` just before it.

In [ ]:
def load_splits():
    # 01_get_data.py download(): the repo has no named config for this task, so the
    # task's two parquet files are loaded directly. Pinned to the commit the local run used.
    files = {s: f"{DATASET_CONFIG}/{s}.parquet" for s in ("train", "test")}
    try:
        from datasets import load_dataset
        ds = load_dataset(DATASET_REPO, data_files=files, revision=DATASET_REVISION)
        return {s: ds[s].to_pandas() for s in files}
    except Exception as exc:        # `datasets` missing or broken by the pin: read the same files directly
        print(f"(datasets.load_dataset failed: {type(exc).__name__}: {str(exc)[:200]}; reading parquet with pandas)")
        from huggingface_hub import hf_hub_download
        return {s: pd.read_parquet(hf_hub_download(DATASET_REPO, f, repo_type="dataset",
                                                   revision=DATASET_REVISION))
                for s, f in files.items()}


def stratified_subsample(labels, n, seed=SEED):
    # phase1_common.py stratified_subsample(), unchanged. n < 0 means "every row".
    labels = np.asarray(labels)
    total = len(labels)
    if n is None or n < 0 or n >= total:
        return np.arange(total)
    from sklearn.model_selection import train_test_split

    idx, _ = train_test_split(np.arange(total), train_size=n,
                              stratify=labels, random_state=seed)
    return np.sort(idx)


def indices_digest(idx):
    # 02_embed.py indices_digest(), unchanged: a short fingerprint of which rows were used
    return hashlib.sha1(np.ascontiguousarray(idx, dtype=np.int64).tobytes()).hexdigest()[:16]


full = load_splits()

# Chromosome split. The dataset's own train/test files ARE the split; this checks it,
# the way phase1_common.py dataset_chromosomes() reads it from the `name` column.
chroms = {s: {n.split(":")[0].removeprefix("chr") for n in full[s]["name"]} for s in full}
assert chroms["train"] <= TRAIN_CHROMS, f"unexpected train chromosomes: {sorted(chroms['train'] - TRAIN_CHROMS)}"
assert chroms["test"] <= TEST_CHROMS, f"unexpected test chromosomes: {sorted(chroms['test'] - TEST_CHROMS)}"
assert not (chroms["train"] & chroms["test"])
print(f"train: {len(full['train'])} rows on {len(chroms['train'])} chromosomes (chr1-19, X, Y)   "
      f"test: {len(full['test'])} rows on chr{', chr'.join(sorted(chroms['test']))}")

idx = {"train": stratified_subsample(full["train"]["label"], N_TRAIN, SEED),
       "test": stratified_subsample(full["test"]["label"], N_TEST, SEED)}
seqs = {s: full[s]["sequence"].iloc[idx[s]].tolist() for s in idx}
y = {s: full[s]["label"].to_numpy()[idx[s]].astype(np.int64) for s in idx}
CLASSES = sorted(LABEL_NAMES)
CLASS_NAMES = [LABEL_NAMES[c] for c in CLASSES]
assert all(set(y[s].tolist()) <= set(CLASSES) for s in y)
if not SMOKE:
    assert all(len(idx[s]) == len(full[s]) for s in idx), "the full run must use every window"
ROWS_KEY = {s: {"n": int(len(idx[s])), "sha1": indices_digest(idx[s])} for s in idx}

# 10% validation rows carved from train: the split 03_train_head.py makes
# (train_test_split(test_size=0.10, stratify=y, random_state=seed)). Every model uses the same rows.
from sklearn.model_selection import train_test_split
fit_rows, val_rows = train_test_split(np.arange(len(y["train"])), test_size=VAL_FRAC,
                                      stratify=y["train"], random_state=HEAD_SEED)
for s in idx:
    print(f"{s:>5}: {len(idx[s])} rows, per class "
          f"{ {LABEL_NAMES[c]: int((y[s] == c).sum()) for c in CLASSES} }")
print(f"train -> {len(fit_rows)} to fit the head + {len(val_rows)} validation")

# Label names re-checked the way 01_get_data.py found them: donors carry GT at offset 301
# (or AC at 298 on the other strand), acceptors AG at 298 (or CT at 301).
def has(seq_list, a, ka, b, kb):
    return float(np.mean([s[ka:ka + 2] == a or s[kb:kb + 2] == b for s in seq_list]))

o = MOTIF_OFFSETS
for c in CLASSES:
    rows = [s for s, lab in zip(seqs["train"], y["train"]) if lab == c]
    d = has(rows, "GT", o["donor_GT_plus"], "AC", o["donor_AC_minus"])
    a = has(rows, "AG", o["acceptor_AG_plus"], "CT", o["acceptor_CT_minus"])
    print(f"label {c} ({LABEL_NAMES[c]:>8}): donor motif {d:.1%}   acceptor motif {a:.1%}")
    if LABEL_NAMES[c] == "donor":
        assert d > 0.9 and a < 0.3, "label 2 does not look like donor"
    if LABEL_NAMES[c] == "acceptor":
        assert a > 0.9 and d < 0.3, "label 1 does not look like acceptor"

## Tokens: which piece of the model's input is the centre?

A model does not read DNA letter by letter. Its **tokenizer** first cuts the 600 letters into pieces called tokens, and the model gives back one vector per token. The label depends on the two letters at the centre of the window, so we want the vector of the token that covers letter 300, plus the token just before it.

Where that token sits is different for every model. The Nucleotide Transformers cut the DNA into 6-letter pieces, so the centre is usually token number 51, but one `N` in the sequence shifts everything. DNABERT-2 cuts pieces of different lengths, so the centre moves from window to window.

So nothing is hard-coded. For **every window**, the function below maps each token back to the letters it covers and checks that the tokens, put back together, spell the original sequence exactly. Then it looks up the token covering letter 300 and the one before it. If any check fails it stops with an error, because a wrong position would not crash: it would quietly read the wrong place and give a worse score.

In [ ]:
from transformers import AutoTokenizer


def load_tokenizer(spec):
    # The tokenizer and the model's settings file (config). Both are tiny downloads.
    from transformers import AutoConfig

    tok = AutoTokenizer.from_pretrained(spec["repo"], revision=spec["revision"], trust_remote_code=True)
    config = AutoConfig.from_pretrained(spec["repo"], revision=spec["revision"], trust_remote_code=True)
    special_ids = np.array(sorted(set(tok.all_special_ids)))
    return tok, special_ids, config


def token_limit(tok, config):
    # 02b_embed_center.py: the smallest of the tokenizer's limit, the model's limit and 2048
    limits = [getattr(tok, "model_max_length", None),
              getattr(config, "max_position_embeddings", None), 2048]
    return int(min(x for x in limits if x and x < 1_000_000))


TOKENIZERS = {}


def token_limit_of(spec, config):
    return token_limit(TOKENIZERS[spec["key"]], config)


def encode_batch(tok, special_ids, max_len, seq_list, seen):
    # Tokenise a batch and find, per sequence, the input position of the token covering base 300
    # (C) and of the token just before it (P).
    # Token -> base mapping from 02b_embed_center.py spans():
    #   fast tokenizer (DNABERT-2): the tokenizer reports each token's [start, end) offsets
    #   slow tokenizer (NT):        cumulative lengths of the token strings
    # Either way the tokens must tile the sequence and spell it exactly.
    # Special tokens are found by id: NT's slow tokenizer returns a special_tokens_mask that
    # assumes an EOS token it never adds (off by one).
    # Previous token = the adjacent token before the centre one, as in 12_repool_center.py.
    enc = tok(seq_list, return_tensors="pt", padding=True, truncation=True, max_length=max_len,
              return_offsets_mapping=bool(tok.is_fast))
    ids, am = enc["input_ids"], enc["attention_mask"]
    ids_np, am_np = ids.numpy(), am.numpy().astype(bool)
    c_idx = np.empty(len(seq_list), dtype=np.int64)
    p_idx = np.empty(len(seq_list), dtype=np.int64)
    for i, seq in enumerate(seq_list):
        body = am_np[i] & ~np.isin(ids_np[i], special_ids)
        pos = np.flatnonzero(body)                     # input positions of the real tokens
        pieces = tok.convert_ids_to_tokens(ids_np[i][body].tolist())
        if tok.is_fast:
            off = enc["offset_mapping"][i].numpy()[body]
            start, end = off[:, 0].astype(np.int64), off[:, 1].astype(np.int64)
            assert start[0] == 0 and end[-1] == len(seq) and (start[1:] == end[:-1]).all(), \
                "token offsets do not tile the sequence"
            assert all(seq[a:b] == p for a, b, p in zip(start, end, pieces)), \
                "tokens do not spell the bases at their offsets"
        else:
            lens = np.array([len(p) for p in pieces])
            assert int(lens.sum()) == len(seq), \
                f"reconstructed length {int(lens.sum())} != sequence length {len(seq)}"
            assert "".join(pieces) == seq, "reconstructed tokens do not spell the sequence"
            end = np.cumsum(lens)
            start = end - lens
        c = np.flatnonzero((start <= CENTER_BASE) & (end > CENTER_BASE))
        assert len(c) == 1, f"expected exactly one token covering base {CENTER_BASE}"
        c = int(c[0])
        assert c >= 1, "no token before the centre token"
        assert end[c - 1] == start[c], "previous token is not adjacent"
        c_idx[i], p_idx[i] = pos[c], pos[c - 1]
        seen[(int(pos[c]), int(start[c]), int(end[c]) - 1,
              int(pos[c - 1]), int(start[c - 1]), int(end[c - 1]) - 1)] += 1
    return ids, am, torch.from_numpy(c_idx), torch.from_numpy(p_idx)


def tokenise_all(spec):
    # Every window of both splits, tokenised once in batches of EMBED_BATCH, with the centre and
    # previous token found and checked. The model is later run on exactly these tensors.
    tok, special_ids, config = load_tokenizer(spec)
    TOKENIZERS[spec["key"]] = tok
    max_len = token_limit(tok, config)
    seen, t0 = collections.Counter(), time.time()
    batches = {split: [encode_batch(tok, special_ids, max_len, seqs[split][i:i + EMBED_BATCH], seen)
                       for i in range(0, len(seqs[split]), EMBED_BATCH)] for split in ("train", "test")}
    centre = centre_summary(seen)
    centre["tokenizer"] = type(tok).__name__ + (" (fast: offsets)" if tok.is_fast else " (slow: token lengths)")
    centre["max_tokens"] = max_len
    centre["tokens_per_window"] = [int(min(int(b[1].sum(1).min()) for s in batches for b in batches[s])),
                                   int(max(int(b[1].sum(1).max()) for s in batches for b in batches[s]))]
    centre["tokenise_seconds"] = round(time.time() - t0, 1)
    return batches, max_len, centre


def centre_summary(seen, top=5):
    # What the lookup found, for results.json and for reading by eye
    total = sum(seen.values())
    rows = [{"rows": n, "share": round(n / total, 4), "centre_input_index": ci, "centre_bases": [cs, ce],
             "previous_input_index": pi, "previous_bases": [ps, pe]}
            for (ci, cs, ce, pi, ps, pe), n in seen.most_common(top)]
    return {"windows_checked": total, "distinct_placements": len(seen), "most_common": rows}

## Running a frozen model

**Frozen** means the model's own numbers never change: it is put in evaluation mode and run without gradients, so this is plain inference. Each window goes through each model exactly once.

**Half precision (fp16)** means the GPU does the arithmetic with 16-bit numbers instead of 32-bit ones. That is about twice as fast and needs less memory. To check it does no harm, the first batch of each model is run both ways and the two results are compared. The 2.5B model is the exception: its weights are also stored in half precision, because the full-precision copy would nearly fill the GPU.

**DNABERT-2 and triton.** DNABERT-2 ships its own model code. When a package called `triton` is installed (it is on Kaggle's GPU image), that code switches to a special fast attention routine that no longer works with current `triton` versions, and the model crashes. The fix used here: hide `triton` while DNABERT-2 loads, then switch that routine off and check it is off. The model then uses its ordinary PyTorch attention, which gives the same numbers.

**If something goes wrong.** Running out of GPU memory halves the batch size and tries again. Any other failure in half precision gets one retry in full precision. If a model still cannot run, it is skipped, the reason is written to `results.json`, and the notebook carries on with the next model.

**Checkpoint.** As soon as a model has embedded every window, its vectors are written to `embeddings/<model>.npz`. If the notebook is run again in the same session, finished models are loaded from disk instead of being computed again.

In [ ]:
@contextlib.contextmanager
def triton_hidden():
    # While this is active, `import triton` fails as if triton were not installed.
    saved = {k: v for k, v in sys.modules.items() if k == "triton" or k.startswith("triton.")}
    for k in saved:
        del sys.modules[k]
    sys.modules["triton"] = None
    try:
        yield
    finally:
        sys.modules.pop("triton", None)
        sys.modules.update(saved)


def switch_off_triton_attention(model):
    # DNABERT-2's bert_layers.py keeps the triton attention routine in a module-level name and
    # falls back to plain PyTorch attention when that name is None. Set it to None and verify.
    changed = 0
    for mod_name in sorted({type(m).__module__ for m in model.modules()}):
        mod = sys.modules.get(mod_name)
        if mod is not None and hasattr(mod, "flash_attn_qkvpacked_func"):
            if mod.flash_attn_qkvpacked_func is not None:
                mod.flash_attn_qkvpacked_func = None
                changed += 1
            assert mod.flash_attn_qkvpacked_func is None
    return changed


def load_model(spec, half_weights):
    # try_dna_models.py load_model(): NT-v2 registers AutoModelForMaskedLM but not AutoModel,
    # so both are tried.
    from transformers import AutoModel, AutoModelForMaskedLM

    kwargs = {"revision": spec["revision"], "trust_remote_code": True}
    if half_weights:
        kwargs["torch_dtype"] = torch.float16
    errors = []
    for cls in (AutoModel, AutoModelForMaskedLM):
        try:
            if spec.get("block_triton"):
                with triton_hidden():
                    model = cls.from_pretrained(spec["repo"], **kwargs)
            else:
                model = cls.from_pretrained(spec["repo"], **kwargs)
            return model, cls.__name__
        except Exception as exc:
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {str(exc)[:300]}")
    raise RuntimeError(" | ".join(errors))


def token_states(model, input_ids, attention_mask):
    # try_dna_models.py get_hidden(): last-layer vector of every token
    out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state
    if getattr(out, "hidden_states", None):
        return out.hidden_states[-1]
    if isinstance(out, (tuple, list)):
        return out[0]
    raise RuntimeError("could not find per-token vectors in the model output")


def centre_features(model, ids, am, c, p, autocast):
    # center_plus_prev_concat: [centre token ++ previous token], in that order
    # (12_repool_center.py: np.concatenate([cen, prev], axis=1)). No gradients: the model is frozen.
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=autocast):
        h = token_states(model, ids.to(DEVICE), am.to(DEVICE))
        r = torch.arange(h.shape[0], device=h.device)
        return torch.cat([h[r, c.to(DEVICE)], h[r, p.to(DEVICE)]], dim=-1).float().cpu().numpy()


def is_out_of_memory(exc):
    return "out of memory" in str(exc).lower()


def free_gpu():
    gc.collect()
    if ON_GPU:
        torch.cuda.empty_cache()


def model_cache_dir(spec):
    from huggingface_hub.constants import HF_HUB_CACHE
    return Path(HF_HUB_CACHE) / ("models--" + spec["repo"].replace("/", "--"))


def weights_downloaded(spec):
    # True if a weight file of this model is already in the download cache
    cache = model_cache_dir(spec)
    return cache.exists() and any(f.is_file() and not f.name.endswith(".incomplete") and f.stat().st_size > 50e6
                                  for f in cache.rglob("*"))


def free_disk_gb(path):
    path = Path(path)
    while not path.exists():
        path = path.parent
    return shutil.disk_usage(path).free / 1e9


def embed_once(spec, batches, max_len, half_weights, autocast):
    # One attempt at embedding every window with one precision setting. Returns (X, info).
    # `batches` are the tokenised windows with their checked centre / previous positions.
    label = ("fp16 weights" if half_weights else "fp32 weights") + (" + fp16 autocast" if autocast else ", no autocast")
    need = spec.get("download_gb", 0) + 2
    if not weights_downloaded(spec) and free_disk_gb(model_cache_dir(spec)) < need:
        raise RuntimeError(f"not enough disk for the download: {free_disk_gb(model_cache_dir(spec)):.1f} GB free, "
                           f"need about {need:.1f} GB")
    if ON_GPU:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    model, loaded_via = load_model(spec, half_weights)
    try:
        model.eval()
        for prm in model.parameters():
            prm.requires_grad_(False)
        triton_switched = switch_off_triton_attention(model) if spec.get("block_triton") else None
        model.to(DEVICE)
        load_s = time.time() - t0
        n_params = sum(prm.numel() for prm in model.parameters())
        hidden = int(model.config.hidden_size)
        assert token_limit_of(spec, model.config) == max_len, "the model's token limit differs from the one used to tokenise"
        print(f"  loaded via {loaded_via} in {load_s:.0f}s ({label}): {n_params / 1e6:.0f}M parameters, "
              f"hidden size {hidden}, head input {2 * hidden}"
              + (f", GPU memory after load {torch.cuda.memory_allocated() / 1e9:.1f} GB" if ON_GPU else ""))

        info = {"precision": label, "loaded_via": loaded_via, "parameters": int(n_params), "hidden_size": hidden,
                "feature_dim": 2 * hidden, "max_tokens": max_len, "load_seconds": round(load_s, 1),
                "triton_attention_switched_off": triton_switched, "fp16_check": None}
        if half_weights:
            info["fp16_check"] = "not run: the weights are fp16, so there is no fp32 result to compare with"
        elif autocast:
            # First 16 windows only: the same windows in full precision and in half precision.
            ids, am, c, p = (t[:16] for t in batches["train"][0])
            f32 = centre_features(model, ids, am, c, p, autocast=False)
            f16 = centre_features(model, ids, am, c, p, autocast=True)
            cos = (f32 * f16).sum(1) / (np.linalg.norm(f32, axis=1) * np.linalg.norm(f16, axis=1))
            info["fp16_check"] = {"windows": int(len(ids)), "min_cosine_vs_fp32": round(float(cos.min()), 6),
                                  "max_abs_diff": round(float(np.abs(f32 - f16).max()), 4)}
            print(f"  fp16 check on the first {len(ids)} windows: cosine with fp32 at least "
                  f"{cos.min():.5f}, largest difference {np.abs(f32 - f16).max():.3f}")
            if not np.isfinite(cos).all() or cos.min() < FP16_MIN_COSINE:
                raise ValueError(f"fp16 output differs from fp32 (min cosine {cos.min():.4f})")

        X, bs = {}, EMBED_BATCH
        n_all = sum(len(b[0]) for split in batches for b in batches[split])
        n_done, t_run, next_report = 0, time.time(), 0.0
        for split in ("train", "test"):
            out = []
            for ids, am, c, p in batches[split]:
                i = 0
                while i < len(ids):
                    try:
                        feats = centre_features(model, ids[i:i + bs], am[i:i + bs], c[i:i + bs], p[i:i + bs], autocast)
                    except RuntimeError as exc:
                        if not is_out_of_memory(exc) or bs <= MIN_BATCH:
                            raise
                        bs = max(MIN_BATCH, bs // 2)
                        free_gpu()
                        print(f"  GPU out of memory: batch size lowered to {bs}, trying again")
                        continue
                    if not np.isfinite(feats).all():
                        raise ValueError(f"non-finite values in the embeddings ({label})")
                    out.append(feats)
                    i += len(feats)
                    n_done += len(feats)
                if time.time() - t_run >= next_report or n_done == n_all:
                    el = time.time() - t_run
                    print(f"  {n_done:6d}/{n_all} windows  {el:6.0f}s elapsed  {n_done / el:6.1f} windows/s  "
                          f"about {(n_all - n_done) * el / n_done:5.0f}s left", flush=True)
                    next_report = el + max(15.0, el * 0.25)
            X[split] = np.concatenate(out)
        if ON_GPU:
            torch.cuda.synchronize()
        embed_s = time.time() - t_run
        info.update({"batch_size_final": bs, "embed_seconds": round(embed_s, 1),
                     "embed_ms_per_window": round(1000 * embed_s / n_all, 2),
                     "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if ON_GPU else None,
                     "peak_gpu_reserved_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2) if ON_GPU else None})
        return X, info
    finally:
        del model
        free_gpu()

In [ ]:
def embedding_path(spec):
    return EMB_DIR / f"{spec['key']}.npz"


def run_key(spec):
    # What a saved embedding file must match to be reused
    return {"repo": spec["repo"], "revision": spec["revision"], "rows": ROWS_KEY, "center_base": CENTER_BASE,
            "dataset_revision": DATASET_REVISION, "features": "center_plus_prev_concat"}


def load_checkpoint(spec):
    path = embedding_path(spec)
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as z:
        meta = json.loads(str(z["meta"]))
        if meta.get("run_key") != json.loads(json.dumps(run_key(spec))):
            print(f"  saved embeddings at {path.name} are for other settings: ignoring them")
            return None
        X = {s: z[f"X_{s}"].astype(np.float32) for s in ("train", "test")}
        assert all(np.array_equal(z[f"y_{s}"], y[s]) for s in y), "saved labels do not match"
    return X, meta["embedding"], meta["centre_token"]


def embed_model(spec):
    # Embeddings for one model: from its checkpoint if there is one, otherwise computed and saved.
    ck = load_checkpoint(spec)
    if ck is not None:
        print(f"  embeddings loaded from checkpoint {embedding_path(spec).name}: not recomputed")
        return ck[0], dict(ck[1], from_checkpoint=True), ck[2]

    # Tokenise and check the centre token in every window BEFORE the model is downloaded:
    # a tokenizer problem should cost seconds, not a 10 GB download.
    batches, max_len, centre = tokenise_all(spec)
    print(f"  centre token found and checked in all {centre['windows_checked']} windows "
          f"({centre['tokenise_seconds']:.0f}s, {centre['tokenizer']}, {centre['tokens_per_window'][0]}-"
          f"{centre['tokens_per_window'][1]} tokens per window); {centre['distinct_placements']} placement(s):")
    for r in centre["most_common"][:3]:
        print(f"    {r['rows']:6d} windows: centre token = input {r['centre_input_index']} (bases "
              f"{r['centre_bases'][0]}-{r['centre_bases'][1]}), previous = input {r['previous_input_index']} "
              f"(bases {r['previous_bases'][0]}-{r['previous_bases'][1]})")

    half = ON_GPU and spec["weights"] == "fp16"
    attempts = [(half, ON_GPU)]                 # (half-precision weights, fp16 autocast)
    if ON_GPU:
        attempts.append((False, False))         # one retry in full precision
    errors = []
    for half_weights, autocast in attempts:
        try:
            X, info = embed_once(spec, batches, max_len, half_weights, autocast)
            break
        except Exception as exc:
            free_gpu()
            errors.append(f"{type(exc).__name__}: {str(exc)[:400]}")
            print(f"  attempt failed: {errors[-1]}")
    else:
        raise RuntimeError(" || ".join(errors))
    del batches
    info["failed_attempts"] = errors
    info["from_checkpoint"] = False
    info["tokenise_seconds"] = centre["tokenise_seconds"]

    # phase1_common.py pair_features(): this variant goes to the head rounded through float16.
    # Stored as float16, which is that same rounding and half the disk space.
    X16 = {s: X[s].astype(np.float16) for s in X}
    assert all(np.isfinite(X16[s]).all() for s in X16), "a value does not fit in float16"
    path, tmp = embedding_path(spec), EMB_DIR / f"{spec['key']}.tmp.npz"
    meta = {"run_key": run_key(spec), "embedding": info, "centre_token": centre, "name": spec["name"]}
    np.savez(tmp, X_train=X16["train"], X_test=X16["test"], y_train=y["train"], y_test=y["test"],
             idx_train=idx["train"], idx_test=idx["test"], meta=np.array(json.dumps(meta)))
    os.replace(tmp, path)
    print(f"  saved {path.name}: train {X16['train'].shape}, test {X16['test'].shape}, "
          f"{path.stat().st_size / 1e6:.0f} MB; embedding took {info['embed_seconds']:.0f}s"
          + (f", peak GPU memory {info['peak_gpu_gb']:.1f} GB" if ON_GPU else ""))
    if ON_KAGGLE and FREE_MODEL_CACHE:
        shutil.rmtree(model_cache_dir(spec), ignore_errors=True)     # the download is no longer needed
    return {s: X16[s].astype(np.float32) for s in X16}, info, centre

## The head, and how it is scored

The head is the only part that learns. It is the same small network for every model: the vector goes into 256 units, a ReLU, dropout of 0.2, and out to 3 scores (none / acceptor / donor). Only its input size changes, because bigger models give longer vectors.

Training follows `03_train_head.py` exactly: inputs are standardised (each column to mean 0, spread 1, measured on the training part only), the optimiser is Adam, and after every pass over the training part the head is scored on the validation part. Training stops once validation has not improved for 20 passes, and the weights from the **best validation pass** are the ones kept. The test windows play no part in any of this.

**The numbers reported.**

- **Macro-F1** is the headline. F1 combines precision and recall for one class; macro-F1 is the plain average of the three classes, so each class counts equally.
- **Accuracy**: the share of test windows labelled correctly.
- **Precision** for a class: of the windows the model *called* donor, how many really are. **Recall**: of the windows that really are donors, how many it found.
- **Confusion matrix**: a 3×3 table of true label against predicted label, which shows what gets mistaken for what.
- **95% interval**: the test set is one sample of about 3,000 windows, so every score has some luck in it. The interval is found by redrawing the test windows 2,000 times with replacement and rescoring. If two models' intervals overlap a lot, do not read much into the gap. The "difference to the reference" interval does this properly, on the same redraws for both models.

In [ ]:
from sklearn.preprocessing import StandardScaler

to_idx = {c: i for i, c in enumerate(CLASSES)}
yi = {s: np.array([to_idx[v] for v in y[s]]) for s in y}


def make_head(in_dim):
    # 03_train_head.py: input -> 256 -> ReLU -> Dropout(0.2) -> classes
    return nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, len(CLASSES)))


def train_mlp(X_fit, y_fit, X_val, y_val):
    # 03_train_head.py train_mlp(): Adam, early stopping on validation loss, best weights restored. On CPU.
    torch.manual_seed(HEAD_SEED)
    gen = torch.Generator().manual_seed(HEAD_SEED)
    Xf, yf = torch.tensor(X_fit, dtype=torch.float32), torch.tensor(y_fit, dtype=torch.long)
    Xv, yv = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)
    model = make_head(Xf.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best = {"val_loss": float("inf"), "epoch": 0, "state": None}
    history, since_best, t0 = [], 0, time.time()
    for epoch in range(1, HEAD_MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xf), generator=gen)
        total = 0.0
        for i in range(0, len(Xf), HEAD_BATCH):
            b = perm[i:i + HEAD_BATCH]
            opt.zero_grad()
            loss = loss_fn(model(Xf[b]), yf[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            logits = model(Xv)
            val_loss = float(loss_fn(logits, yv))
            val_acc = float((logits.argmax(1) == yv).float().mean())
        history.append({"epoch": epoch, "train_loss": round(total / len(Xf), 5),
                        "val_loss": round(val_loss, 5), "val_acc": round(val_acc, 4),
                        "elapsed_s": round(time.time() - t0, 2)})
        if val_loss < best["val_loss"] - 1e-4:
            best = {"val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
            since_best = 0
        else:
            since_best += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"    epoch {epoch:3d}  train loss {total / len(Xf):.4f}  val loss {val_loss:.4f}  "
                  f"val acc {val_acc:.4f}  elapsed {time.time() - t0:.1f}s")
        if since_best >= HEAD_PATIENCE:
            print(f"    early stop at epoch {epoch} (no val-loss improvement for {HEAD_PATIENCE} epochs); "
                  f"keeping epoch {best['epoch']}  elapsed {time.time() - t0:.1f}s")
            break
    model.load_state_dict(best["state"])
    model.eval()
    return model, best, history


def confusion(y_true_i, y_pred_i):
    # rows = true class, columns = predicted class (class positions 0..2)
    k = len(CLASSES)
    return np.bincount(y_true_i * k + y_pred_i, minlength=k * k).reshape(k, k)


def macro_f1_from(cm):
    tp = np.diag(cm).astype(float)
    denom = cm.sum(0) + cm.sum(1)               # 2*TP + FP + FN per class
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))


def evaluate(y_true, y_pred):
    # 03_train_head.py evaluate(): accuracy, macro-F1, per-class precision/recall/F1, confusion matrix
    from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                                 precision_recall_fscore_support)

    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    return {
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)), 4),
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "per_class": {LABEL_NAMES[c]: {"label": int(c), "precision": round(float(p[i]), 4),
                                       "recall": round(float(r[i]), 4), "f1": round(float(f[i]), 4),
                                       "support": int(s[i])} for i, c in enumerate(CLASSES)},
        "confusion_matrix": {"rows": "true label", "cols": "predicted label", "names": CLASS_NAMES,
                             "matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist()},
    }


# The same redraws of the test windows for every model, so models can be compared pair by pair.
BOOT_ROWS = np.random.default_rng(SEED).integers(0, len(y["test"]), size=(N_BOOT, len(y["test"])))


def bootstrap(pred_i):
    # macro-F1 and accuracy of each redraw
    true_i = yi["test"]
    f1 = np.empty(N_BOOT)
    acc = np.empty(N_BOOT)
    for b, rows in enumerate(BOOT_ROWS):
        cm = confusion(true_i[rows], pred_i[rows])
        f1[b], acc[b] = macro_f1_from(cm), np.trace(cm) / cm.sum()
    return f1, acc


def interval(values):
    lo, hi = np.percentile(values, [2.5, 97.5])
    return [round(float(lo), 4), round(float(hi), 4)]


def train_and_score(spec, X):
    # Standardise, train the head, score the test windows once.
    scaler = StandardScaler().fit(X["train"][fit_rows])
    t0 = time.time()
    head, best, hist = train_mlp(scaler.transform(X["train"][fit_rows]), yi["train"][fit_rows],
                                 scaler.transform(X["train"][val_rows]), yi["train"][val_rows])
    head_s = time.time() - t0
    with torch.no_grad():
        pred_i = head(torch.tensor(scaler.transform(X["test"]), dtype=torch.float32)).argmax(1).numpy()
    pred = np.array(CLASSES)[pred_i]
    m = evaluate(y["test"], pred)
    assert abs(macro_f1_from(confusion(yi["test"], pred_i)) - m["macro_f1"]) < 1e-4    # my F1 == sklearn's F1
    f1_b, acc_b = bootstrap(pred_i)
    m["macro_f1_ci95"], m["accuracy_ci95"] = interval(f1_b), interval(acc_b)
    training = {
        "features": "center_plus_prev_concat, rounded through float16, StandardScaler fit on the fit rows",
        "head": f"Linear({X['train'].shape[1]},256) -> ReLU -> Dropout(0.2) -> Linear(256,{len(CLASSES)})",
        "optimizer": f"Adam(lr={HEAD_LR}, weight_decay={HEAD_WEIGHT_DECAY})", "batch_size": HEAD_BATCH,
        "max_epochs": HEAD_MAX_EPOCHS, "patience": HEAD_PATIENCE, "selection": "lowest validation loss",
        "n_fit": int(len(fit_rows)), "n_val": int(len(val_rows)), "seed": HEAD_SEED,
        "best_epoch": best["epoch"], "epochs_run": len(hist),
        "best_val_loss": round(best["val_loss"], 5), "best_val_acc": round(best["val_acc"], 4),
        "train_seconds": round(head_s, 1), "history": hist,
    }
    np.savez(PRED_DIR / f"{spec['key']}.npz", y_true=y["test"], y_pred=pred, dataset_row=idx["test"])
    torch.save({"state_dict": head.state_dict(), "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_,
                "classes": CLASSES, "label_names": LABEL_NAMES, "in_dim": int(X["train"].shape[1])},
               PRED_DIR / f"{spec['key']}_head.pt")
    return m, training, f1_b


def print_report(title, m):
    print(f"  --- {title} ---")
    print(f"  test macro-F1 {m['macro_f1']:.4f}  (95% interval {m['macro_f1_ci95'][0]:.3f}-{m['macro_f1_ci95'][1]:.3f})"
          f"   accuracy {m['accuracy']:.4f}")
    for name, v in m["per_class"].items():
        print(f"  {name:>10}  precision {v['precision']:.3f}  recall {v['recall']:.3f}  n {v['support']}")
    print("  confusion matrix (rows = true, columns = predicted): " + " / ".join(CLASS_NAMES))
    for name, row in zip(CLASS_NAMES, m["confusion_matrix"]["matrix"]):
        print(f"  {name:>10}  " + "  ".join(f"{v:5d}" for v in row))

## Figures

Two kinds of picture. The **confusion matrix** (one per model) shows, for each true class, where its windows ended up; a good model has nearly everything on the diagonal. The **comparison chart** puts the models side by side on macro-F1, with the 95% interval as a line through each dot. The axis is zoomed in on the range where the scores are, so read the numbers, not the distances from the left edge.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

SURFACE, INK, INK_2, GRID, BLUE = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1", "#2a78d6"
BLUE_RAMP = LinearSegmentedColormap.from_list("blue_ramp", [SURFACE, "#cde2fb", "#86b6ef", "#3987e5",
                                                           "#1c5cab", "#0d366b"])
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
                     "xtick.color": INK_2, "ytick.color": INK_2})
SMOKE_TAG = "SMOKE RUN, not a result.  " if SMOKE else ""


def show(fig, path):
    fig.savefig(path, dpi=160, bbox_inches="tight")
    if "inline" in matplotlib.get_backend().lower():
        plt.show()
    plt.close(fig)


def plot_confusion(spec, m):
    # Colour = share of the true class (row); each cell also shows the count.
    cm = np.array(m["confusion_matrix"]["matrix"])
    share = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(4.6, 4.3))
    ax.imshow(share, cmap=BLUE_RAMP, vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}\n{share[i, j]:.1%}", ha="center", va="center", fontsize=10,
                    color="white" if share[i, j] > 0.55 else INK)
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_xticks(np.arange(-0.5, cm.shape[1]), minor=True)
    ax.set_yticks(np.arange(-0.5, cm.shape[0]), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)
    ax.set_title(f"{spec['name']}: test confusion matrix\n{SMOKE_TAG}macro-F1 {m['macro_f1']:.3f}, "
                 f"accuracy {m['accuracy']:.3f}, {int(cm.sum()):,} windows", fontsize=10, loc="left", color=INK)
    show(fig, FIG_DIR / f"confusion_{spec['key']}.png")


def plot_comparison(done):
    # One dot per model: test macro-F1 with its 95% interval, best at the top.
    done = sorted(done, key=lambda r: r["test"]["macro_f1"])
    fig, ax = plt.subplots(figsize=(8.6, 1.3 + 0.62 * len(done)))
    lo = min(r["test"]["macro_f1_ci95"][0] for r in done)
    hi = max(r["test"]["macro_f1_ci95"][1] for r in done)
    pad = max(0.01, 0.12 * (hi - lo))
    for k, r in enumerate(done):
        t, e = r["test"], r["embedding"]
        ax.plot(t["macro_f1_ci95"], [k, k], color=BLUE, linewidth=2, solid_capstyle="round")
        ax.plot([t["macro_f1"]], [k], "o", color=BLUE, markersize=9, markeredgecolor=SURFACE, markeredgewidth=2)
        ax.text(t["macro_f1"], k + 0.24, f"{t['macro_f1']:.3f}", ha="center", va="bottom", fontsize=10, color=INK)
        cost = f"accuracy {t['accuracy']:.3f}   embedding {e['embed_seconds'] / 60:.1f} min"
        if e.get("peak_gpu_gb") is not None:
            cost += f"   peak GPU {e['peak_gpu_gb']:.1f} GB"
        ax.text(1.02, k, cost, transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=9, color=INK_2)
    ax.set_yticks(range(len(done)), [f"{r['name']}\n{r['embedding']['parameters'] / 1e6:,.0f}M parameters" for r in done])
    ax.set_ylim(-0.6, len(done) - 0.3)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_xlabel("test macro-F1 (dot) with 95% bootstrap interval (line); axis does not start at 0")
    ax.grid(axis="x", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    n_test = sum(v["support"] for v in done[0]["test"]["per_class"].values())
    ax.set_title(f"{SMOKE_TAG}Frozen DNA language models on splice sites: same head, {n_test:,} test windows (chr20/21)",
                 fontsize=11, loc="left", color=INK)
    show(fig, FIG_DIR / "comparison.png")

## Run every model

This is the cell that takes the time. For each model, in order: find and check the centre token in every window, embed every window, save the vectors, train the head, score it on the test windows, draw its confusion matrix, and update `results.json`. Because `results.json` and the embedding files are written after **each** model, a crash or a timeout loses at most the model that was running.

In [ ]:
def write_json_atomic(path, obj):
    # phase1_common.py write_json_atomic(): via a temp file, so a crash never leaves half a file
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
        fh.write("\n")
    os.replace(tmp, path)


results = {
    "smoke": SMOKE,
    "note": ("SMOKE run: tiny sample, one model. These numbers only show the notebook runs."
             if SMOKE else "Full run: every training and test window."),
    "task": "3-class splice-site windows (none / acceptor / donor), frozen backbone + small MLP head",
    "dataset": f"{DATASET_REPO} :: {DATASET_CONFIG}", "dataset_revision": DATASET_REVISION,
    "split": {"train_chromosomes": "chr1-19, X, Y", "test_chromosomes": "chr20, chr21",
              "n_train": int(len(y["train"])), "n_fit": int(len(fit_rows)), "n_val": int(len(val_rows)),
              "n_test": int(len(y["test"])), "val_split_seed": HEAD_SEED, "rows": ROWS_KEY},
    "features": "last-layer vector of the token covering base 300, concatenated with the token before it",
    "device": str(DEVICE), "gpu": GPU_NAME, "gpu_total_gb": round(GPU_TOTAL_GB, 1) if ON_GPU else None,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__},
    "reference_model": REFERENCE, "bootstrap_resamples": N_BOOT,
    "run_order": [m["key"] for m in RUN_MODELS], "models": {},
}
BOOT_F1 = {}
for n_model, spec in enumerate(RUN_MODELS, start=1):
    t_model = time.time()
    print(f"\n===== {n_model}/{len(RUN_MODELS)}  {spec['name']}  ({spec['repo']}) =====")
    entry = {"name": spec["name"], "repo": spec["repo"], "revision": spec["revision"]}
    try:
        X, emb_info, centre = embed_model(spec)
        print(f"  training the head on {len(fit_rows)} windows, validating on {len(val_rows)}")
        metrics, training, BOOT_F1[spec["key"]] = train_and_score(spec, X)
        entry.update({"status": "ok", "embedding": emb_info, "centre_token": centre,
                      "head_training": training, "test": metrics})
        print_report(f"{spec['name']}: frozen backbone + head (n={len(y['test'])})", metrics)
        plot_confusion(spec, metrics)
        del X
    except Exception as exc:
        entry.update({"status": "skipped", "reason": f"{type(exc).__name__}: {str(exc)[:1500]}"})
        print(f"  SKIPPED {spec['name']}: {entry['reason'][:600]}")
    free_gpu()
    entry["seconds_total"] = round(time.time() - t_model, 1)
    results["models"][spec["key"]] = entry
    results["total_seconds"] = round(time.time() - T_START, 1)
    write_json_atomic(RESULTS_JSON, results)
    print(f"  {spec['name']} done in {entry['seconds_total'] / 60:.1f} min; results.json updated "
          f"({(time.time() - T_START) / 60:.1f} min since the start)")

## Compare, and pack everything for download

The table and the chart rank the models by test macro-F1. The last column of the table is each model's gap to the reference model (NT-v2 50M) with its own 95% interval: if that interval does not include 0, the difference is unlikely to be luck.

Two zip files are written next to the output folder:

- `compare_models_results.zip` (small): `results.json`, the figures, each model's test predictions and trained head.
- `compare_models_embeddings.zip` (large, several hundred MB): the saved vectors of every model. With these, other head sizes and depths can be tried later on the laptop, with no GPU.

In [ ]:
done = [dict(results["models"][m["key"]], key=m["key"]) for m in RUN_MODELS
        if results["models"][m["key"]]["status"] == "ok"]
skipped = [results["models"][m["key"]] for m in RUN_MODELS if results["models"][m["key"]]["status"] != "ok"]

# Difference to the reference model, on the same redraws of the test windows
for r in done:
    if REFERENCE in BOOT_F1 and r["key"] != REFERENCE:
        diff = BOOT_F1[r["key"]] - BOOT_F1[REFERENCE]
        results["models"][r["key"]]["macro_f1_minus_reference"] = r["macro_f1_minus_reference"] = {
            "reference": REFERENCE,
            "difference": round(r["test"]["macro_f1"] - results["models"][REFERENCE]["test"]["macro_f1"], 4),
            "ci95": interval(diff)}
results["total_seconds"] = round(time.time() - T_START, 1)
write_json_atomic(RESULTS_JSON, results)

if SMOKE:
    print("SMOKE RUN: tiny sample, one model. Not a result. Set SMOKE = False for the real numbers.\n")
print(f"{'model':<16}{'macro-F1':>9}{'95% interval':>16}{'accuracy':>10}" + "".join(f"{'recall ' + n:>17}" for n in CLASS_NAMES)
      + f"{'embed min':>11}{'peak GPU GB':>13}   macro-F1 minus reference")
for r in sorted(done, key=lambda r: -r["test"]["macro_f1"]):
    t, e = r["test"], r["embedding"]
    d = r.get("macro_f1_minus_reference")
    gpu = f"{e['peak_gpu_gb']:.1f}" if e.get("peak_gpu_gb") is not None else "-"
    print(f"{r['name']:<16}{t['macro_f1']:>9.4f}{t['macro_f1_ci95'][0]:>9.3f}-{t['macro_f1_ci95'][1]:.3f}{t['accuracy']:>10.4f}"
          + "".join(f"{t['per_class'][n]['recall']:>17.4f}" for n in CLASS_NAMES)
          + f"{e['embed_seconds'] / 60:>11.1f}{gpu:>13}   "
          + (f"{d['difference']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]" if d else "(reference)" if r["key"] == REFERENCE else "-"))
for r in skipped:
    print(f"{r['name']:<16}SKIPPED: {r['reason'][:200]}")

if done:
    plot_comparison(done)


def zip_folder(zip_path, files, compress):
    with zipfile.ZipFile(zip_path, "w", compress) as z:
        for f in files:
            z.write(f, Path(OUT_DIR.name) / f.relative_to(OUT_DIR))
    return zip_path.stat().st_size / 1e6


small = [RESULTS_JSON] + sorted(FIG_DIR.glob("*.png")) + sorted(PRED_DIR.glob("*"))
big = sorted(EMB_DIR.glob("*.npz"))
z1 = BASE_DIR / f"{OUT_DIR.name}_results.zip"
z2 = BASE_DIR / f"{OUT_DIR.name}_embeddings.zip"
print(f"\nwrote {z1.name}: {len(small)} files, {zip_folder(z1, small, zipfile.ZIP_DEFLATED):.1f} MB")
print(f"wrote {z2.name}: {len(big)} files, {zip_folder(z2, big, zipfile.ZIP_STORED):.0f} MB")
print(f"\n{len(done)} of {len(RUN_MODELS)} models finished, {len(skipped)} skipped.   device: {DEVICE}   "
      f"total time: {(time.time() - T_START) / 60:.1f} min")